# Data Split

Loads the processed XSum dataset from `data/processed/xsum_processed.jsonl`
and splits it into **80% train / 10% validation / 10% test** sets, saved
back to:

- `data/processed/train.jsonl`
- `data/processed/validation.jsonl`
- `data/processed/test.jsonl`


## 0. Setup — Imports and Paths

In [1]:
import os
import json
import random

import pandas as pd

PROCESSED_DIR = "data/processed"
INPUT_PATH = os.path.join(PROCESSED_DIR, "xsum_processed.jsonl")

TRAIN_PATH = os.path.join(PROCESSED_DIR, "train.jsonl")
VAL_PATH = os.path.join(PROCESSED_DIR, "validation.jsonl")
TEST_PATH = os.path.join(PROCESSED_DIR, "test.jsonl")

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

SEED = 42
random.seed(SEED)

print(f"Input file:      {INPUT_PATH}")
print(f"Train output:    {TRAIN_PATH}")
print(f"Validation output: {VAL_PATH}")
print(f"Test output:      {TEST_PATH}")


Input file:      data/processed\xsum_processed.jsonl
Train output:    data/processed\train.jsonl
Validation output: data/processed\validation.jsonl
Test output:      data/processed\test.jsonl


## 1. Load the Processed Dataset

Reads `data/processed/xsum_processed.jsonl` (one JSON record per line)
into a DataFrame.


In [2]:

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"Could not find '{INPUT_PATH}'. Run the preprocessing notebook first "
        "to generate the processed dataset."
    )

records = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

processed_df = pd.DataFrame(records)
print(f"Loaded {len(processed_df)} processed records from '{INPUT_PATH}'")
processed_df.head()


Loaded 1000 processed records from 'data/processed\xsum_processed.jsonl'


,article_id,article,summary,chunk,chunk_count,prompt,prompt_word_count,prompt_token_length,input_ids
0,34577037,Sam Ward also found the net but two goals from...,Great Britain were beaten 3-2 in the first of ...,[Sam Ward also found the net but two goals fro...,1,[You are a news simplification assistant. Simp...,[189],[273],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
1,35794084,A blue Seat Leon car was involved in a collisi...,A teenage boy has died after a crash on the A9...,[A blue Seat Leon car was involved in a collis...,1,[You are a news simplification assistant. Simp...,[297],[412],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
2,20266195,"Soares, 26, joined the Potters in 2008 but mad...",Bury have signed former Stoke City midfielder ...,"[Soares, 26, joined the Potters in 2008 but ma...",1,[You are a news simplification assistant. Simp...,[222],[313],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
3,34551854,"This is ""to ensure it was thorough, properly c...",Another police force has been asked to review ...,"[This is ""to ensure it was thorough, properly ...",1,[You are a news simplification assistant. Simp...,[630],[780],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
4,34769588,Simon Lee Bell was taken to hospital after an ...,The family of a 22-year-old man who died follo...,[Simon Lee Bell was taken to hospital after an...,1,[You are a news simplification assistant. Simp...,[195],[268],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."


## 2. Shuffle and Split into Train / Validation / Test

The dataset is shuffled with a fixed seed for reproducibility, then split
80% / 10% / 10%. Row counts are computed so all rows are assigned even
when the dataset size doesn't divide evenly (any remainder goes to train).


In [3]:
assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-9, \
    "Split ratios must sum to 1.0"

# Shuffle indices reproducibly
shuffled_df = processed_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n = len(shuffled_df)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)
# Remaining rows (handles rounding) go to test
n_test = n - n_train - n_val

train_df = shuffled_df.iloc[:n_train].reset_index(drop=True)
val_df = shuffled_df.iloc[n_train:n_train + n_val].reset_index(drop=True)
test_df = shuffled_df.iloc[n_train + n_val:].reset_index(drop=True)

print(f"Total: {n}")
print(f"Train:      {len(train_df)} ({len(train_df) / n:.1%})")
print(f"Validation: {len(val_df)} ({len(val_df) / n:.1%})")
print(f"Test:       {len(test_df)} ({len(test_df) / n:.1%})")

Total: 1000
Train:      800 (80.0%)
Validation: 100 (10.0%)
Test:       100 (10.0%)


## 3. Save the Splits to `data/processed`

Each split is written as JSONL (one record per line) to its own file.


In [4]:
def save_jsonl(df: pd.DataFrame, path: str):
    with open(path, "w", encoding="utf-8") as f:
        for record in df.to_dict(orient="records"):
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved {len(df)} records to '{path}'")


save_jsonl(train_df, TRAIN_PATH)
save_jsonl(val_df, VAL_PATH)
save_jsonl(test_df, TEST_PATH)

Saved 800 records to 'data/processed\train.jsonl'
Saved 100 records to 'data/processed\validation.jsonl'
Saved 100 records to 'data/processed\test.jsonl'
